[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farhad-abtahi/healthcareaibook/blob/main/vol%201%20notebooks/chapter_04/notebook_4_1_data_quality_assessment.ipynb)

*Click the badge above to open this notebook in Google Colab (no setup required)*

---


# Notebook 4.1: Data Quality Assessment in Healthcare

**Chapter 4: Data Preparation**

## Learning Objectives

By the end of this notebook, you will be able to:
1. Assess data completeness, consistency, and validity
2. Detect outliers and anomalous values
3. Generate comprehensive data quality reports
4. Visualize data quality issues systematically
5. Understand how poor data quality impacts model performance

## Clinical Context

Real-world healthcare data is messy:
- Missing values (lab not ordered, equipment failure)
- Inconsistencies (different units, typos)
- Outliers (data entry errors, rare true values)
- Temporal issues (delayed documentation)

**"Garbage in, garbage out"** applies doubly in healthcare where bad predictions harm patients.

---

## Setup

In [1]:
# ── Setup (JupyterLite) ─────────────────────────────────────────────────────
import sys
if sys.platform == "emscripten":          # JupyterLite / Pyodide
    import micropip
    await micropip.install(["seaborn"])

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

print("✓ Libraries imported")

✓ Libraries imported


## 1. Generate Realistic Messy Healthcare Data

In [3]:
def generate_messy_ehr_data(n_patients=1000):
    """Generate realistic EHR data with quality issues.

    Each patient first gets TRUE physiological values, and the outcome is drawn from those
    (with randomness, like real risk). Only then are recording errors injected into what the
    chart shows. So the label reflects the patient, and the errors are noise a model must cope with.
    """
    data = []

    for i in range(n_patients):
        # Where and when the record was created (quality differs by site and era)
        site = np.random.choice(['Site A', 'Site B', 'Site C'], p=[0.40, 0.35, 0.25])
        admit_year = int(np.random.choice([2019, 2020, 2021, 2022, 2023]))

        # TRUE patient state (what is really going on)
        age_true = float(np.clip(np.random.normal(65, 15), 18, 100))
        glucose_true = float(np.clip(np.random.normal(120, 30), 50, 400))   # mg/dL
        sbp_true = np.random.normal(130, 20)
        sex_true = np.random.choice(['M', 'F'])

        # Outcome (e.g. a complication) drawn from the TRUE values, before any recording errors
        logit = -1.2 + 0.05 * (age_true - 65) + 0.03 * (glucose_true - 120) + 0.01 * (sbp_true - 130)
        outcome = int(np.random.rand() < 1 / (1 + np.exp(-logit)))

        # --- What the chart records ---
        # Age: 15% missing
        age = age_true if np.random.rand() >= 0.15 else np.nan

        # Sex: same truth, inconsistent coding (and 2% missing)
        sex = None if np.random.rand() < 0.02 else np.random.choice(
            {'M': ['M', 'Male', 'm'], 'F': ['F', 'Female', 'f']}[sex_true], p=[0.7, 0.2, 0.1])

        # Labs with missing values (30% missing; 50% at Site C, whose lab interface drops results)
        p_missing = 0.50 if site == 'Site C' else 0.30
        glucose = glucose_true if np.random.rand() > p_missing else np.nan

        # Errors: sentinels (10% before the 2021 EHR upgrade added range checks, 2% after),
        # and unit errors (8%: value entered in mmol/L instead of mg/dL)
        p_error = 0.10 if admit_year < 2021 else 0.02
        if not np.isnan(glucose):
            if np.random.rand() < p_error:
                glucose = float(np.random.choice([999, -1, 9999]))  # Data entry errors
            elif np.random.rand() < 0.08:
                glucose = glucose_true / 18                          # mmol/L entered as mg/dL

        # Blood pressure - sometimes recorded inconsistently
        if np.random.rand() > 0.20:
            sbp = sbp_true
            # Different formats
            if np.random.rand() < 0.7:
                bp = f"{int(sbp)}/{int(sbp * 0.6)}"
            else:
                bp = int(sbp)  # Only systolic
        else:
            bp = None

        # Medications - inconsistent naming
        if np.random.rand() < 0.4:
            meds = np.random.choice(['Metformin', 'metformin', 'METFORMIN',
                                    'Metformin 500mg', None])
        else:
            meds = None

        data.append({
            'patient_id': f'P{i:04d}',
            'site': site,
            'admit_year': admit_year,
            'age': age,
            'sex': sex,
            'glucose': glucose,
            'bp': bp,
            'medications': meds,
            'outcome': outcome
        })

    return pd.DataFrame(data)

df = generate_messy_ehr_data(1000)
print(f"Generated {len(df)} synthetic patient records")
df.head(10)

Generated 1000 synthetic patient records


,patient_id,site,admit_year,age,sex,glucose,bp,medications,outcome
0,P0000,Site A,2023,73.156147,Male,NaN,106,Metformin 500mg,1
1,P0001,Site A,2022,97.124876,M,120.666655,121/72,NaN,1
2,P0002,Site A,2020,55.996192,M,148.423195,135/81,NaN,0
3,P0003,Site A,2020,55.466604,f,126.559149,147,NaN,0
4,P0004,Site A,2023,73.731842,Male,146.632454,147/88,NaN,0
5,P0005,Site A,2021,76.324967,F,NaN,128/77,NaN,0
6,P0006,Site A,2019,66.376412,M,60.372933,125/75,NaN,0
7,P0007,Site A,2022,70.356689,F,999.000000,149/89,NaN,1
8,P0008,Site B,2023,48.556013,f,147.446530,116/70,Metformin,0
9,P0009,Site C,2022,57.293188,F,NaN,136/81,NaN,0


## 2. Data Quality Assessment

In [4]:
class DataQualityAssessor:
    """Comprehensive data quality assessment."""

    def __init__(self, df):
        self.df = df
        self.report = {}

    def assess_completeness(self):
        """Check missing values."""
        missing = self.df.isnull().sum()
        missing_pct = (missing / len(self.df)) * 100

        self.report['completeness'] = pd.DataFrame({
            'missing_count': missing,
            'missing_pct': missing_pct
        }).sort_values('missing_pct', ascending=False)

        return self.report['completeness']

    def assess_consistency(self, column):
        """Check value consistency."""
        if column not in self.df.columns:
            return None

        values = self.df[column].dropna()
        value_counts = values.value_counts()

        # Check for similar values (e.g., 'M' vs 'Male')
        unique_values = set(str(v).lower() for v in values.unique())

        return {
            'unique_values': len(value_counts),
            'value_counts': value_counts,
            'normalized_unique': len(unique_values)
        }

    def detect_outliers(self, column, method='iqr', threshold=3):
        """Detect outliers in numeric columns."""
        if column not in self.df.columns:
            return None

        values = pd.to_numeric(self.df[column], errors='coerce').dropna()

        if method == 'iqr':
            Q1 = values.quantile(0.25)
            Q3 = values.quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            outliers = (values < lower) | (values > upper)
        else:  # z-score
            z_scores = np.abs(stats.zscore(values))
            outliers = z_scores > threshold

        return {
            'n_outliers': outliers.sum(),
            'outlier_pct': (outliers.sum() / len(values)) * 100,
            'outlier_values': values[outliers].tolist()
        }

    def generate_report(self):
        """Generate comprehensive quality report."""
        print("="*80)
        print("DATA QUALITY ASSESSMENT REPORT")
        print("="*80)

        # Dataset overview
        print(f"\nDataset: {len(self.df)} rows, {len(self.df.columns)} columns")

        # Completeness
        print("\n1. COMPLETENESS")
        print("-" * 80)
        completeness = self.assess_completeness()
        print(completeness)

        # Consistency for categorical columns
        print("\n2. CONSISTENCY (Categorical)")
        print("-" * 80)
        for col in ['sex', 'medications']:
            if col in self.df.columns:
                consistency = self.assess_consistency(col)
                if consistency:
                    print(f"\n{col}:")
                    print(f"  Unique values: {consistency['unique_values']}")
                    print(f"  Top values:\n{consistency['value_counts'].head()}")

        # Outliers in numeric columns
        print("\n3. OUTLIERS (Numeric)")
        print("-" * 80)
        for col in ['age', 'glucose']:
            if col in self.df.columns:
                outliers = self.detect_outliers(col)
                if outliers:
                    print(f"\n{col}:")
                    print(f"  Outliers: {outliers['n_outliers']} ({outliers['outlier_pct']:.1f}%)")
                    if outliers['outlier_values']:
                        print(f"  Examples: {outliers['outlier_values'][:5]}")

        print("\n" + "="*80)

# Run assessment
assessor = DataQualityAssessor(df)
assessor.generate_report()

DATA QUALITY ASSESSMENT REPORT

Dataset: 1000 rows, 9 columns

1. COMPLETENESS
--------------------------------------------------------------------------------
             missing_count  missing_pct
medications            663         66.3
glucose                362         36.2
bp                     206         20.6
age                    152         15.2
sex                     22          2.2
patient_id               0          0.0
admit_year               0          0.0
site                     0          0.0
outcome                  0          0.0

2. CONSISTENCY (Categorical)
--------------------------------------------------------------------------------



sex:
  Unique values: 6
  Top values:
sex
M         360
F         345
Female    106
Male       93
m          40
Name: count, dtype: int64

medications:
  Unique values: 4
  Top values:
medications
metformin          92
Metformin 500mg    88
METFORMIN          84
Metformin          73
Name: count, dtype: int64

3. OUTLIERS (Numeric)
--------------------------------------------------------------------------------

age:
  Outliers: 6 (0.7%)
  Examples: [26.8475272556679, 26.173750516937005, 22.504545647858897, 26.435207858672513, 18.0]

glucose:
  Outliers: 81 (12.7%)
  Examples: [999.0, 4.891948078561657, 8.77818524864437, 6.744967656274571, 4.901310796851748]



## 3. Visualize Data Quality Issues

In [5]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Missing value heatmap
ax = axes[0, 0]
missing_matrix = df.isnull()
sns.heatmap(missing_matrix.iloc[:100], cbar=False, yticklabels=False, ax=ax)
ax.set_title('Missing Data Pattern (First 100 rows)', fontweight='bold')

# Missing percentage by column
ax = axes[0, 1]
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_pct.plot(kind='barh', ax=ax, color='coral')
ax.set_xlabel('Missing %')
ax.set_title('Missing Data by Column', fontweight='bold')
ax.grid(True, alpha=0.3)

# Glucose distribution with outliers
ax = axes[1, 0]
glucose_clean = pd.to_numeric(df['glucose'], errors='coerce').dropna()
ax.hist(glucose_clean[glucose_clean < 500], bins=50, alpha=0.7, color='steelblue')
ax.axvline(glucose_clean.median(), color='red', linestyle='--', label=f'Median: {glucose_clean.median():.0f}')
ax.set_xlabel('Glucose (mg/dL)')
ax.set_title('Glucose Distribution (excluding extreme outliers)', fontweight='bold')
ax.legend()

# Age box plot with outliers
ax = axes[1, 1]
age_clean = df['age'].dropna()
ax.boxplot(age_clean, vert=False)
ax.set_xlabel('Age (years)')
ax.set_title('Age Distribution (Box Plot)', fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3b. Compare Data Quality Across Subgroups and Time Periods

Overall percentages can hide a site or an era with much worse data. Stratify the same checks by `site` and `admit_year`.
These differences were built into the synthetic generator above, so this section shows how to *find* them.

In [6]:
glucose_num = pd.to_numeric(df['glucose'], errors='coerce')
df_q = df.assign(
    age_missing=df['age'].isna(),
    glucose_missing=glucose_num.isna(),
    bp_missing=df['bp'].isna(),
    # Implausible glucose = sentinel / data-entry error (impossible as mg/dL: <= 0 or > 800)
    glucose_implausible=(glucose_num <= 0) | (glucose_num > 800),
    # Ambiguous low values (< 54 mg/dL): either an mmol/L entry or genuine hypoglycaemia
    # (< 54 mg/dL is ADA level 2, clinically significant). Flag for review; check the lab's unit field.
    glucose_unit_suspect=(glucose_num > 0) & (glucose_num < 54),
)
quality_cols = ['age_missing', 'glucose_missing', 'bp_missing', 'glucose_implausible', 'glucose_unit_suspect']

for group_col in ['site', 'admit_year']:
    table = df_q.groupby(group_col)[quality_cols].mean().mul(100).round(1)
    table.insert(0, 'n', df_q.groupby(group_col).size())
    print(f"\nDATA QUALITY BY {group_col.upper()} (% of records)")
    print(table.to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df_q.groupby('site')[['age_missing', 'glucose_missing', 'bp_missing']].mean().mul(100).plot(kind='bar', ax=axes[0])
axes[0].set_ylabel('Missing %'); axes[0].set_title('Missingness by Site', fontweight='bold')
df_q.groupby('admit_year')['glucose_implausible'].mean().mul(100).plot(marker='o', ax=axes[1], color='coral')
axes[1].set_ylabel('Implausible glucose %'); axes[1].set_title('Data-Entry Errors by Admission Year', fontweight='bold')
plt.tight_layout()
plt.show()

# Findings computed from the tables above
by_site = df_q.groupby('site')['glucose_missing'].mean()
worst, best = by_site.idxmax(), by_site.idxmin()
print(f"\n💡 Glucose missingness ranges from {by_site[best]:.0%} ({best}) to {by_site[worst]:.0%} ({worst}).")
if by_site[worst] - by_site[best] > 0.10:
    print(f"   A model trained mostly on {best} data will see far more imputed glucose values at {worst}.")
early = df_q[df_q['admit_year'] < 2021]['glucose_implausible'].mean()
late = df_q[df_q['admit_year'] >= 2021]['glucose_implausible'].mean()
print(f"💡 Implausible glucose values: {early:.1%} of records before 2021 vs {late:.1%} from 2021 on.")
if early > 2 * late:
    print("   Data quality changed over time: older records need stricter cleaning, and a model")
    print("   validated only on recent data may not reflect the training data's error rate.")
else:
    print("   No large change over time in this sample.")


DATA QUALITY BY SITE (% of records)
          n  age_missing  glucose_missing  bp_missing  glucose_implausible  glucose_unit_suspect
site                                                                                            
Site A  390         13.8             30.0        19.0                  5.4                   3.6
Site B  352         17.3             29.5        19.9                  3.4                   7.1
Site C  258         14.3             54.7        24.0                  2.7                   2.3

DATA QUALITY BY ADMIT_YEAR (% of records)
              n  age_missing  glucose_missing  bp_missing  glucose_implausible  glucose_unit_suspect
admit_year                                                                                          
2019        189         15.3             37.6        19.0                  9.0                   4.2
2020        208         18.8             37.5        17.8                  5.8                   2.9
2021        182         14.8   

## 4. Impact on Model Performance

In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# The outcome was generated from the TRUE values (see the generator), so cleaning can now matter.
y = df['outcome']
# One split, used by every model below: same training rows, same test rows
train_idx, test_idx = train_test_split(np.arange(len(df)), test_size=0.3, random_state=42, stratify=y)
glucose_raw = pd.to_numeric(df['glucose'], errors='coerce')
age_median = df['age'].iloc[train_idx].median()

def clean_glucose(g):
    """
    Sentinels → missing. Low values are ambiguous: 2-30 could be mmol/L (normal) or mg/dL
    (severe hypoglycaemia). Real pipelines use the lab system's unit field and send ambiguous
    values for review. SHORTCUT valid only for this synthetic generator, which never produces a
    true glucose below 50 mg/dL: convert only values that are implausible as mg/dL (< 20) yet
    plausible as mmol/L (>= 2), and set the remaining ambiguous values (20-50) to missing.
    """
    g = g.copy()
    g[(g < 2) | (g >= 900)] = np.nan                    # sentinels (-1, 999, 9999) and impossible values
    mmol = (g >= 2) & (g < 20)                         # implausible as mg/dL, plausible as mmol/L
    ambiguous = (g >= 20) & (g < 50)                   # could be either: do not guess
    g[mmol] = g[mmol] * 18
    g[ambiguous] = np.nan
    return g

print(f"Glucose values converted from mmol/L: {int(((glucose_raw >= 2) & (glucose_raw < 20)).sum())}, "
      f"ambiguous (20-50) set to missing for review: {int(((glucose_raw >= 20) & (glucose_raw < 50)).sum())}")

glucose_clean = clean_glucose(glucose_raw)
X_dirty = pd.DataFrame({'age': df['age'].fillna(age_median), 'glucose': glucose_raw.fillna(120)})
X_clean = pd.DataFrame({'age': df['age'].fillna(age_median),
                        'glucose': glucose_clean.fillna(glucose_clean.iloc[train_idx].median())})

print("="*60)
print("IMPACT OF DATA QUALITY ON MODEL PERFORMANCE")
print("="*60)
print(f"Outcome prevalence: {y.mean():.1%}; test set: {len(test_idx)} patients (cleaned features for both models)\n")
results = {}
for name, model in [('Logistic regression', LogisticRegression(max_iter=1000)),
                    ('Random forest', RandomForestClassifier(n_estimators=100, random_state=42))]:
    auc_dirty = roc_auc_score(y.iloc[test_idx], model.fit(X_dirty.iloc[train_idx], y.iloc[train_idx])
                              .predict_proba(X_clean.iloc[test_idx])[:, 1])
    auc_clean = roc_auc_score(y.iloc[test_idx], model.fit(X_clean.iloc[train_idx], y.iloc[train_idx])
                              .predict_proba(X_clean.iloc[test_idx])[:, 1])
    results[name] = (auc_dirty, auc_clean)
    print(f"{name:20s} trained on DIRTY data: AUROC {auc_dirty:.3f} | on CLEANED data: AUROC {auc_clean:.3f} "
          f"| gain {auc_clean - auc_dirty:+.3f}")
print("="*60)

lr_gain = results['Logistic regression'][1] - results['Logistic regression'][0]
rf_gain = results['Random forest'][1] - results['Random forest'][0]
if lr_gain > 0.02:
    print(f"\n💡 Cleaning helped the logistic regression by {lr_gain:+.3f} AUROC: sentinels like 9999 and")
    print("   mmol/L values distort the fitted glucose effect for every patient, not just the bad rows.")
else:
    print(f"\n💡 Cleaning changed logistic-regression AUROC by only {lr_gain:+.3f} in this sample.")
if abs(rf_gain) < 0.02:
    print(f"   The random forest barely changed ({rf_gain:+.3f}): trees can isolate extreme values in their own")
    print("   branches, so they are more robust to sentinels. Robust models hide data problems; they do not fix them.")
else:
    print(f"   The random forest changed by {rf_gain:+.3f}.")

Glucose values converted from mmol/L: 41, ambiguous (20-50) set to missing for review: 0
IMPACT OF DATA QUALITY ON MODEL PERFORMANCE
Outcome prevalence: 27.1%; test set: 300 patients (cleaned features for both models)

Logistic regression  trained on DIRTY data: AUROC 0.570 | on CLEANED data: AUROC 0.673 | gain +0.102


Random forest        trained on DIRTY data: AUROC 0.620 | on CLEANED data: AUROC 0.640 | gain +0.020

💡 Cleaning helped the logistic regression by +0.102 AUROC: sentinels like 9999 and
   mmol/L values distort the fitted glucose effect for every patient, not just the bad rows.
   The random forest changed by +0.020.


## 5. Key Takeaways

### What We Learned

1. **Completeness**: Check missing value patterns
2. **Consistency**: Standardize formats and values
3. **Validity**: Detect outliers and impossible values
4. **Impact**: Poor data quality degrades model performance; in Section 4 cleaning glucose helped the
   logistic regression far more than the random forest, because trees can isolate sentinel values on their own
5. **Stratify**: Quality differs by site and time period; overall averages can hide a bad subgroup
6. **Process**: Always assess quality before modeling

### Real-World Considerations

- EHR data is inherently messy
- Missing data is rarely random: here glucose missingness depends on the site (Section 3b); in practice it is often MNAR
- Outliers may be errors OR rare true values
- Documentation varies by provider
- Data quality assessment is ongoing

## Exercises

1. **IQR vs z-score**: `DataQualityAssessor.detect_outliers` (Section 2) uses the IQR rule by default. Call it with `method='zscore'` for `glucose` and `age`. Which method flags more glucose values, and why do the 9999 sentinels inflate the standard deviation the z-score depends on?

2. **Normalise the categories**: Write a small mapping that collapses `sex` to `M`/`F` and `medications` to a single `metformin` label, then run `DataQualityAssessor(df_fixed).generate_report()` on the fixed copy. How many unique values remain, and how many `medications` entries hide a dose (`500mg`) you would want as a separate column?

3. **Use the blood pressure column**: In Section 4 the models use only `age` and `glucose`, but the outcome was generated from systolic BP too. Parse `bp` (formats `"138/82"` and `138`) into an `sbp` feature, impute it with the training median, and add it to `X_dirty` and `X_clean`. Does the AUROC rise, and does the dirty-vs-clean gap change?

4. **Site shift**: Using `df_q` from Section 3b, train the cleaned logistic regression on Sites A and B only and test on Site C, where half of the glucose values are missing. Compare the AUROC with the random split in Section 4 and look at how many Site C test patients receive the imputed median glucose.

5. **Reflective**: The cleaning function `clean_glucose` relies on a shortcut that is only valid because the generator never produces a true glucose below 50 mg/dL. In a real hospital, what information (lab unit field, reference range, clinician review) would you need before converting or discarding a glucose of 4.5?

---

*This notebook is part of "AI in Healthcare" (Volume 1, Chapter 4: Data Preparation)*